# Customer Segmentation Using Machine Learning
End-to-end notebook for synthetic data generation, cleaning, EDA, clustering, profiling, RFM analysis, and business recommendations.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'src'))
from data_preprocessing import generate_synthetic_customer_data, preprocess_pipeline
from clustering import kmeans_diagnostics, fit_kmeans, fit_hierarchical, fit_dbscan, choose_best_model, metrics_table, hierarchical_linkage_sample, pca_projection
from insights import build_cluster_profiles, attach_segment_names, generate_business_insights
from visualization import save_eda_charts, save_model_charts, save_segment_charts


## Generate Data, Clean, and Engineer Features

In [ ]:
raw = generate_synthetic_customer_data(n_records=6000, random_state=42)
prep = preprocess_pipeline(raw)
prep.cleaning_report, prep.outlier_report


## Model Training and Evaluation
StandardScaler is used because distance-based clustering can be dominated by high-scale variables such as income and annual spending.

In [ ]:
diagnostics = kmeans_diagnostics(prep.scaled_features)
# K=2 has the highest raw silhouette score, but a post-binary K gives more useful marketing segments.
optimal_k = int(diagnostics[diagnostics['k'].between(3, 6)].sort_values('silhouette', ascending=False).iloc[0]['k'])
results = [fit_kmeans(prep.scaled_features, optimal_k), fit_hierarchical(prep.scaled_features, optimal_k), fit_dbscan(prep.scaled_features)]
comparison = metrics_table(results)
best = choose_best_model(results)
comparison


## Segment Profiles and Insights

In [ ]:
profiles = build_cluster_profiles(prep.feature_data, best.labels)
segmented = attach_segment_names(prep.feature_data, profiles, best.labels)
profiles


In [ ]:
for i, insight in enumerate(generate_business_insights(segmented, profiles), start=1):
    print(f'{i}. {insight}')


## Save Visualizations and Deliverables

In [ ]:
chart_dir = PROJECT_ROOT / 'outputs' / 'charts'
output_dir = PROJECT_ROOT / 'outputs'
(PROJECT_ROOT / 'data').mkdir(exist_ok=True)
output_dir.mkdir(exist_ok=True)
raw.to_csv(PROJECT_ROOT / 'data' / 'customer_data.csv', index=False)
segmented[['Customer_ID', 'Assigned_Segment', 'Cluster']].to_csv(output_dir / 'customer_segments.csv', index=False)
segmented.to_csv(output_dir / 'customer_segments_full.csv', index=False)
profiles.to_csv(output_dir / 'cluster_profiles.csv', index=False)
comparison.to_csv(output_dir / 'model_comparison.csv', index=False)
prep.outlier_report.to_csv(output_dir / 'outlier_comparison.csv', index=False)
pca_data = pca_projection(prep.scaled_features, best.labels)
linkage_matrix = hierarchical_linkage_sample(prep.scaled_features)
save_eda_charts(segmented, chart_dir)
save_model_charts(diagnostics, linkage_matrix, prep.scaled_features, best.labels, pca_data, chart_dir)
save_segment_charts(segmented, chart_dir)
print('Deliverables exported.')
